# SUR Estimator - Exercise Solutions

Solutions for exercises in `advanced/02_seemingly_unrelated_regressions.ipynb`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import panelbox as pb

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

---
## Exercise 1: SUR with Independent Errors

Simulate a panel with independent errors and verify that the BP test does NOT reject and efficiency gains are close to 1.0.

In [ ]:
# Simulate panel with independent errors
N, T = 5, 50

rows = []
for i in range(1, N + 1):
    alpha_i = np.random.normal(0, 3)
    beta_i = np.random.normal(2, 0.5)
    for t in range(1, T + 1):
        x = np.random.normal(0, 2)
        eps = np.random.normal(0, 1)  # independent errors
        y = alpha_i + beta_i * x + eps
        rows.append({'entity': i, 'time': t, 'y': y, 'x': x})

indep_data = pd.DataFrame(rows)

# Fit SUR
sur_indep = pb.SUR('y ~ x', indep_data, 'entity', 'time')
res_indep = sur_indep.fit()

# BP test
bp = res_indep.bp_independence_test
print("Breusch-Pagan Independence Test")
print(f"LM statistic: {bp['statistic']:.4f}")
print(f"p-value:      {bp['pvalue']:.4f}")
print(f"Reject at 5%: {bp['pvalue'] < 0.05}")
print()

# Efficiency gains
entities = list(res_indep.entity_params.keys())
print("Efficiency gains (SE_SUR / SE_OLS):")
for e in entities:
    gains = res_indep.efficiency_gain[e]
    for var in gains.index:
        print(f"  Entity {e}, {var}: {gains[var]:.4f}")

print("\nWith independent errors, the BP test should not reject,")
print("and efficiency gains should be close to 1.0 (no improvement).")

---
## Exercise 2: ISUR Convergence Speed

Compare ISUR with different tolerances on the Grunfeld data.

In [ ]:
data = pb.load_grunfeld()
entities = sorted(data['firm'].unique())

tolerances = [1e-4, 1e-8, 1e-12]
results = {}

for tol in tolerances:
    sur = pb.SUR('invest ~ value + capital', data, 'firm', 'year',
                  iterate=True, max_iter=500, tol=tol)
    res = sur.fit()
    results[tol] = res
    print(f"tol={tol:.0e}: {res.n_iterations} iterations, converged={res.converged}")

print()

# Compare coefficients for first entity
first_entity = entities[0]
print(f"Coefficients for entity {first_entity}:")
print(f"{'Variable':<12} {'tol=1e-4':>12} {'tol=1e-8':>12} {'tol=1e-12':>12} {'Max diff':>10}")
print("-" * 60)

vars_list = results[tolerances[0]].entity_params[first_entity].index
for var in vars_list:
    vals = [results[t].entity_params[first_entity][var] for t in tolerances]
    max_diff = max(vals) - min(vals)
    print(f"{var:<12} {vals[0]:>12.6f} {vals[1]:>12.6f} {vals[2]:>12.6f} {max_diff:>10.2e}")

print("\nTighter tolerance requires more iterations but coefficients")
print("converge quickly - differences are typically negligible.")

---
## Exercise 3: Homogeneity Test with Homogeneous DGP

Simulate a panel where all entities share the same true slope, then test homogeneity.

In [ ]:
# Simulate with homogeneous slopes but correlated errors
N, T = 6, 50
true_beta = 2.0

# Correlated errors
rho = 0.5
Sigma = (1 - rho) * np.eye(N) + rho * np.ones((N, N))
L = np.linalg.cholesky(Sigma)
eps = (L @ np.random.randn(N, T)).T  # (T, N)

rows = []
for i in range(N):
    alpha_i = np.random.normal(0, 3)
    for t in range(T):
        x = np.random.normal(0, 2)
        y = alpha_i + true_beta * x + eps[t, i]
        rows.append({'entity': i + 1, 'time': t + 1, 'y': y, 'x': x})

hom_data = pd.DataFrame(rows)

# Fit unrestricted SUR
sur_hom = pb.SUR('y ~ x', hom_data, 'entity', 'time')
res_hom = sur_hom.fit()

# LR homogeneity test
lr = sur_hom.lr_test_homogeneity()

print("LR Test for Slope Homogeneity")
print("=" * 40)
print(f"LR statistic: {lr['statistic']:.4f}")
print(f"p-value:      {lr['pvalue']:.4f}")
print(f"df:           {lr['df']}")
print(f"Reject H0:    {lr['reject_h0']}")
print()

# Show entity-specific slopes
print(f"True beta: {true_beta:.1f}")
print("Estimated entity-specific slopes:")
for e in sorted(res_hom.entity_params.keys()):
    print(f"  Entity {e}: {res_hom.entity_params[e]['x']:.4f}")

print("\nSince the true DGP has homogeneous slopes, the LR test")
print("should generally fail to reject H0 (p-value > 0.05).")